In [1]:
from relbench.datasets import get_dataset, get_dataset_names, register_dataset

## Hyper Data

In [ ]:
import os
import pandas as pd
import numpy as np
from relbench.base import Database, Dataset, Table

class TransactionalDataset(Dataset):
    # Set timestamps or other relevant information if needed
    val_timestamp = pd.Timestamp("2022-02-15")
    test_timestamp = pd.Timestamp("2022-02-22")

    def make_db(self) -> Database:
        # Path to your CSVs folder
        path = os.path.join("D:/Dani/relbench/relbench/", "hyper_data")
        customers = os.path.join(path, "Customers.csv")
        articles = os.path.join(path, "Articles.csv")
        branches = os.path.join(path, "Branches.csv")
        transactions = os.path.join(path, "Transactions.csv")

        # Ensure that CSV files exist in the specified path
        if not os.path.exists(customers):
            raise RuntimeError(f"Dataset not found at '{path}'. Please make sure the CSV files are in the correct folder.")

        # Read the CSV data into pandas DataFrames
        customers_df = pd.read_csv(customers)
        articles_df = pd.read_csv(articles)
        branches_df = pd.read_csv(branches)
        transactions_df = pd.read_csv(transactions)
        transactions_df['d_dat'] = pd.to_datetime(transactions_df['d_dat'])        
        split_date = pd.to_datetime('2021-08-01')
        transactions_df = transactions_df[transactions_df['d_dat'] >= split_date]
        transactions_df = transactions_df.reset_index(drop=True)
        ################################################################################
        # Check for and handle duplicate primary keys in articles, customers, and branches tables
        ################################################################################

        # Handle duplicates in the articles table
        if articles_df.duplicated(subset=['articles_id']).any():
            print("Duplicates found in the 'articles_id' column. Removing duplicates...")
            articles_df = articles_df.drop_duplicates(subset=['articles_id'], keep='first')

        # Handle duplicates in the customers table
        if customers_df.duplicated(subset=['customers_id']).any():
            print("Duplicates found in the 'customers_id' column. Removing duplicates...")
            customers_df = customers_df.drop_duplicates(subset=['customers_id'], keep='first')

        # Handle duplicates in the branches table
        if branches_df.duplicated(subset=['BranchCode']).any():
            print("Duplicates found in the 'BranchCode' column. Removing duplicates...")
            branches_df = branches_df.drop_duplicates(subset=['BranchCode'], keep='first')

        ################################################################################
        # Clean and process the data (drop unnecessary columns, handle missing data)
        ################################################################################
        # Drop unnecessary columns
        transactions_df.drop(columns=["Return Amount"], inplace=True)
        articles_df.drop(columns=["Item Barcode", "External Item Number"], inplace=True)

        # Replace any missing or invalid values
        transactions_df["salesTime"] = transactions_df["salesTime"].replace(r"^\\N$", "00:00:00", regex=True)
        transactions_df = transactions_df.replace(r"^\\N$", np.nan, regex=True)

        # Combine date and time into a single 'datetime' column
        # transactions_df['datetime'] = pd.to_datetime(transactions_df['d_dat'] + ' ' + transactions_df['salesTime'])
        # transactions_df.drop(columns=["d_dat"], inplace=True)        
        # Convert date column to pd.Timestamp
        # transactions_df["datetime"] = pd.to_datetime(transactions_df["datetime"])

        transactions_df["datetime"] = pd.to_datetime(
        transactions_df["d_dat"], format="%Y-%m-%d"
        )
        transactions_df.drop(columns=["d_dat"], inplace=True)          
        # Convert other fields if necessary
        transactions_df['price_purchase'] = pd.to_numeric(transactions_df['price_purchase'], errors='coerce')
        transactions_df['Discount_ratio'] = pd.to_numeric(transactions_df['Discount_ratio'], errors='coerce')
        transactions_df['Quantity'] = pd.to_numeric(transactions_df['Quantity'], errors='coerce')

        ################################################################################
        # Now we define the table structure and relationships.
        ################################################################################

        tables = {}

        # Articles table
        tables["article"] = Table(
            df=pd.DataFrame(articles_df),
            fkey_col_to_pkey_table={},
            pkey_col="articles_id",
            time_col=None,
        )

        # Customers table
        tables["customer"] = Table(
            df=pd.DataFrame(customers_df),
            fkey_col_to_pkey_table={},
            pkey_col="customers_id",
            time_col=None,
        )

        # Branches table (renamed from "branche" to "branches")
        tables["branches"] = Table(
            df=pd.DataFrame(branches_df),
            fkey_col_to_pkey_table={},
            pkey_col="BranchCode",
            time_col=None,
        )

        # Transactions table
        tables["transactions"] = Table(
            df=pd.DataFrame(transactions_df),
            fkey_col_to_pkey_table={
                "articles_id": "article",    # Foreign key to articles
                "customers_id": "customer",  # Foreign key to customers
                "BranchCode": "branches",    # Foreign key to branches
            },
            pkey_col=None,
            time_col="datetime",  # Use the combined datetime column for time-based operations
        )

        return Database(tables)


## BurgerLand Data

In [56]:
import os
import pandas as pd
import numpy as np
from relbench.base import Database, Dataset, Table

class TransactionalDataset(Dataset):
    # Set timestamps or other relevant information if needed
    val_timestamp = pd.Timestamp("2024-12-01")
    test_timestamp = pd.Timestamp("2024-12-02")

    def make_db(self) -> Database:
        # Path to your CSVs folder
        path = os.path.join("D:/Dani/relbench/relbench/burger_data/", "burger")   
        customers = os.path.join(path, "customer_data.csv")
        articles = os.path.join(path, "article_data.csv")
        branches = os.path.join(path, "branch_data.csv")
        transactions = os.path.join(path, "transaction_data.csv")

        # Ensure that CSV files exist in the specified path
        if not os.path.exists(customers):
            raise RuntimeError(f"Dataset not found at '{path}'. Please make sure the CSV files are in the correct folder.")

        # Read the CSV data into pandas DataFrames
        customers_df = pd.read_csv(customers)
        articles_df = pd.read_csv(articles)
        branches_df = pd.read_csv(branches)
        transactions_df = pd.read_csv(transactions)
        transactions_df['d_dat'] = pd.to_datetime(transactions_df['d_dat'])
        split_date = pd.to_datetime('2022-01-01')
        transactions_df = transactions_df[transactions_df['d_dat'] >= split_date]
        transactions_df = transactions_df.reset_index(drop=True)
        ################################################################################
        # Check for and handle duplicate primary keys in articles, customers, and branches tables
        ################################################################################

        # Handle duplicates in the articles table
        if articles_df.duplicated(subset=['articles_id']).any():
            print("Duplicates found in the 'articles_id' column. Removing duplicates...")
            articles_df = articles_df.drop_duplicates(subset=['articles_id'], keep='first')

        # Handle duplicates in the customers table
        if customers_df.duplicated(subset=['customers_id']).any():
            print("Duplicates found in the 'customers_id' column. Removing duplicates...")
            customers_df = customers_df.drop_duplicates(subset=['customers_id'], keep='first')

        # Handle duplicates in the branches table
        if branches_df.duplicated(subset=['BranchCode']).any():
            print("Duplicates found in the 'BranchCode' column. Removing duplicates...")
            branches_df = branches_df.drop_duplicates(subset=['BranchCode'], keep='first')

        ################################################################################
        # Clean and process the data (drop unnecessary columns, handle missing data)
        ################################################################################
        # Drop unnecessary columns
        transactions_df.drop(columns=["factor_id"], inplace=True)
        transactions_df.drop(columns=["date"], inplace=True)
        transactions_df.drop(columns=["art_name"], inplace=True)
        transactions_df.drop(columns=["price_item"], inplace=True)
        transactions_df.drop(columns=["BranchName"], inplace=True)
        transactions_df.drop(columns=["FoodTypeName"], inplace=True)
        transactions_df.drop(columns=["FoodType"], inplace=True)
        transactions_df.drop(columns=["Maliat"], inplace=True)       
        transactions_df.drop(columns=["Service"], inplace=True)            
        transactions_df.drop(columns=["group_id"], inplace=True)            
        # articles_df.drop(columns=["Item Barcode", "External Item Number"], inplace=True)

        transactions_df['Sales channel'] = pd.factorize(transactions_df['Sales channel'])[0]

        # Encoding Payment channel column
        # transactions_df['week'] = pd.factorize(transactions_df['week'])[0]


        # Replace any missing or invalid values
        # transactions_df["salesTime"] = transactions_df["salesTime"].replace(r"^\\N$", "00:00:00", regex=True)
        transactions_df = transactions_df.replace(r"^\\N$", np.nan, regex=True)

        # Combine date and time into a single 'datetime' column
        # transactions_df['datetime'] = pd.to_datetime(transactions_df['d_dat'] + ' ' + transactions_df['salesTime'])
        # transactions_df.drop(columns=["d_dat"], inplace=True)        
        # Convert date column to pd.Timestamp
        # transactions_df["datetime"] = pd.to_datetime(transactions_df["datetime"])

        transactions_df["datetime"] = pd.to_datetime(
        transactions_df["d_dat"], format="%Y-%m-%d"
        )
        transactions_df.drop(columns=["d_dat"], inplace=True)          
        # Convert other fields if necessary
        transactions_df['price_purchase'] = pd.to_numeric(transactions_df['price_purchase'], errors='coerce')
        transactions_df['takhfif'] = pd.to_numeric(transactions_df['takhfif'], errors='coerce')
        transactions_df['Quantity'] = pd.to_numeric(transactions_df['Quantity'], errors='coerce')

        ################################################################################
        # Now we define the table structure and relationships.
        ################################################################################

        tables = {}

        # Articles table
        tables["article"] = Table(
            df=pd.DataFrame(articles_df),
            fkey_col_to_pkey_table={},
            pkey_col="articles_id",
            time_col=None,
        )

        # Customers table
        tables["customer"] = Table(
            df=pd.DataFrame(customers_df),
            fkey_col_to_pkey_table={},
            pkey_col="customers_id",
            time_col=None,
        )

        # Branches table (renamed from "branche" to "branches")
        tables["branches"] = Table(
            df=pd.DataFrame(branches_df),
            fkey_col_to_pkey_table={},
            pkey_col="BranchCode",
            time_col=None,
        )

        # Transactions table
        tables["transactions"] = Table(
            df=pd.DataFrame(transactions_df),
            fkey_col_to_pkey_table={
                "articles_id": "article",    # Foreign key to articles
                "customers_id": "customer",  # Foreign key to customers
                "BranchCode": "branches",    # Foreign key to branches
            },
            pkey_col=None,
            time_col="datetime",  # Use the combined datetime column for time-based operations
        )

        return Database(tables)


In [57]:
transactional_dataset = TransactionalDataset()
db = transactional_dataset.make_db()

Duplicates found in the 'articles_id' column. Removing duplicates...


In [58]:
table = db.table_dict["transactions"]

In [59]:
db.table_dict["transactions"]

Table(df=
         SaleTime     customers_id  articles_id  Quantity  takhfif  NetAmount  \
0        01:01:07       9125464221          901       1.0      0.0    95000.0   
1        01:01:37       9123096989          103       2.0      0.0   201000.0   
2        01:03:27       9121835587          103       2.0      0.0   204549.0   
3        01:03:27       9121835587          501       1.0      0.0    51426.0   
4        01:03:27       9121835587          612       1.0      0.0    16526.0   
...           ...              ...          ...       ...      ...        ...   
5165543  21:31:14  995555555597731          209       1.0      0.0   255881.0   
5165544  22:35:56  995555555597664          311       1.0   8675.0   202000.0   
5165545  23:22:19  995555555597642          114       2.0      0.0   697500.0   
5165546  14:35:41  995555555598063          501       1.0   5375.0   116183.0   
5165547  22:23:03       9124062285          103       1.0      0.0   277780.0   

               Nu

In [6]:
db.table_dict["branches"]

Table(df=
    BranchCode          BranchName
0        13810             جم سنتر
1        13805         سعادت آباد 
2        13808           باغ فردوس
3        13807         هايپر استار
4        13804             اندرزگو
5        13802           تهرانپارس
6        13809                کورش
7        13813             ستارخان
8        13812            هديش مال
9        13811               بيهقي
10       13814  شعبه بازار (پرنسا)
11       13803            پاسداران
12       13800            کال سنتر,
  fkey_col_to_pkey_table={},
  pkey_col=BranchCode,
  time_col=None)

In [19]:
db.table_dict["article"]

Table(df=
      articles_id         art_name  price_item FoodTypeName  group_id
0             901          هات داگ     71500.0      هات داگ       900
1             103        جوسي برگر     88500.0       برگرها       100
2             501  سيب زميني مخصوص     44500.0    پيش غذاها       500
3             612  اسپرايت خانواده     14300.0   نوشيدني ها       600
4             105        آيلي برگر     88500.0       برگرها       100
...           ...              ...         ...          ...       ...
5249          812     بيکن اِگ رَپ    129500.0       صبحانه       800
5250          814     چيکن اِگ رَپ    154500.0       صبحانه       800
5251          813  هات داگ اِگ رَپ    124500.0       صبحانه       800
5252          811  هات داگ املت رپ    137500.0       صبحانه       800
5259          810     بيکن املت رپ    134500.0       صبحانه       800

[124 rows x 5 columns],
  fkey_col_to_pkey_table={},
  pkey_col=articles_id,
  time_col=None)

In [ ]:
db.table_dict["customer"]

In [7]:
table.df.iloc[table.df["datetime"].idxmax()]


SaleTime                     14:04:54
customers_id          995555555687627
articles_id                       209
Quantity                          1.0
takhfif                           0.0
NetAmount                    269000.0
Num                         1156144.0
Sales channel                       0
BranchCode                      13811
price_purchase               220500.0
Total_Quantity                    1.0
basketprice                  269000.0
datetime          2024-12-03 00:00:00
Name: 5147695, dtype: object

In [8]:
table.df.iloc[table.df["datetime"].idxmin()]

SaleTime                     01:01:07
customers_id               9125464221
articles_id                       901
Quantity                          1.0
takhfif                           0.0
NetAmount                     95000.0
Num                          555417.0
Sales channel                       0
BranchCode                      13810
price_purchase                71500.0
Total_Quantity                    1.0
basketprice                   95000.0
datetime          2022-03-22 01:01:07
Name: 0, dtype: object

In [60]:
register_dataset("burger-aras6666666", TransactionalDataset)
get_dataset_names()

['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-trial',
 'burger-aras',
 'burger-aras666666',
 'burger-aras6666666']

In [61]:
hyper_dataset = get_dataset("burger-aras6666666")
hyper_dataset

TransactionalDataset()

In [62]:
hyper_dataset.val_timestamp, hyper_dataset.test_timestamp

(Timestamp('2024-12-01 00:00:00'), Timestamp('2024-12-02 00:00:00'))

In [24]:
import relbench

relbench.__version__

'1.1.0'

In [64]:
import duckdb
import pandas as pd
from relbench.tasks import get_task, get_task_names, register_task
from relbench.base import Database, EntityTask, RecommendationTask, Table, TaskType
from relbench.metrics import (
    accuracy,
    average_precision,
    f1,
    link_prediction_map,
    link_prediction_precision,
    link_prediction_recall,
    mae,
    r2,
    rmse,
    roc_auc,
)
from metrics import link_prediction_top
class UserItemPurchaseTask(RecommendationTask):
    r"""Predict the list of articles each customer will purchase in the next seven
    days."""

    task_type = TaskType.LINK_PREDICTION
    src_entity_col = "customer_id"
    src_entity_table = "customer"
    dst_entity_col = "article_id"
    dst_entity_table = "article"
    time_col = "timestamp"
    timedelta = pd.Timedelta(days=7)
    metrics = [link_prediction_precision, link_prediction_recall, link_prediction_map, link_prediction_top]
    eval_k = 12

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        customer = db.table_dict["customer"].df
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})

        df = duckdb.sql(
            f"""
            SELECT
                t.timestamp,
                transactions.customer_id,
                LIST(DISTINCT transactions.article_id) AS article_id
            FROM
                timestamp_df t
            LEFT JOIN
                transactions
            ON
                transactions.t_dat > t.timestamp AND
                transactions.t_dat <= t.timestamp + INTERVAL '{self.timedelta} days'
            GROUP BY
                t.timestamp,
                transactions.customer_id
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={
                self.src_entity_col: self.src_entity_table,
                self.dst_entity_col: self.dst_entity_table,
            },
            pkey_col=None,
            time_col=self.time_col,
        )

# Task 1: Predict articles each customer will purchase in the next 7 days
class CustomerArticlePurchaseTask(RecommendationTask):
    r"""Predict the list of articles each customer will purchase in the next seven days."""
    
    task_type = TaskType.LINK_PREDICTION
    src_entity_col = "customers_id"
    src_entity_table = "customer"
    dst_entity_col = "articles_id"
    dst_entity_table = "article"
    time_col = "timestamp"
    timedelta = pd.Timedelta(days=7)
    metrics = [link_prediction_precision, link_prediction_recall, link_prediction_map, link_prediction_top]
    eval_k = 4

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})

        df = duckdb.sql(
            f"""
            SELECT
                t.timestamp,
                transactions.customers_id,
                LIST(DISTINCT transactions.articles_id) AS articles_id
            FROM
                timestamp_df t
            LEFT JOIN
                transactions
            ON
                transactions.datetime > t.timestamp AND
                transactions.datetime <= t.timestamp + INTERVAL '{self.timedelta.days} days'
            GROUP BY
                t.timestamp,
                transactions.customers_id
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={
                self.src_entity_col: self.src_entity_table,
                self.dst_entity_col: self.dst_entity_table,
            },
            pkey_col=None,
            time_col=self.time_col,
        )


# Task 2: Predict customer churn (no purchases in the next week)
class CustomerChurnTask(EntityTask):
    r"""Predict the churn for a customer (no transactions) in the next 6 days."""

    task_type = TaskType.BINARY_CLASSIFICATION
    entity_col = "customers_id"
    entity_table = "customer"
    time_col = "timestamp"
    target_col = "churn"
    timedelta = pd.Timedelta(days=7)
    metrics = [average_precision, accuracy, f1, roc_auc]

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        customer = db.table_dict["customer"].df
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})

        df = duckdb.sql(
            f"""
            SELECT
                timestamp,
                customers_id,
                CAST(
                    NOT EXISTS (
                        SELECT 1
                        FROM transactions
                        WHERE
                            transactions.customers_id = customer.customers_id AND
                            transactions.datetime > timestamp AND
                            transactions.datetime <= timestamp + INTERVAL '{self.timedelta}'
                    ) AS INTEGER
                ) AS churn
            FROM
                timestamp_df,
                customer
            WHERE
                EXISTS (
                    SELECT 1
                    FROM transactions
                    WHERE
                        transactions.customers_id = customer.customers_id AND
                        transactions.datetime > timestamp - INTERVAL '{self.timedelta}' AND
                        transactions.datetime <= timestamp
                )
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={self.entity_col: self.entity_table},
            pkey_col=None,
            time_col=self.time_col,
        ) 



# Task 3: Predict article sales in the next 7 days
class ItemSalesTask(EntityTask):
    r"""Predict the total sales for an article (the sum of prices of the associated
    transactions) in the next week."""

    task_type = TaskType.REGRESSION
    entity_col = "articles_id"
    entity_table = "article"
    time_col = "timestamp"
    target_col = "sales"
    timedelta = pd.Timedelta(days=7)
    metrics = [r2, mae, rmse]

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})
        article = db.table_dict["article"].df

        df = duckdb.sql(
            f"""
            SELECT
                timestamp,
                articles_id,
                sales
            FROM
                timestamp_df,
                article,
                (
                    SELECT
                        COALESCE(SUM(price_purchase), 0) AS sales
                    FROM
                        transactions
                    WHERE
                        transactions.articles_id = article.articles_id AND
                        datetime > timestamp AND
                        datetime <= timestamp + INTERVAL '{self.timedelta}'
                )
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={"articles_id": "article"},
            pkey_col=None,
            time_col="timestamp",
        )


class ItemQuantityTask(EntityTask):
    r"""Predict the total quantity of purchased products for an article in the next day."""

    task_type = TaskType.REGRESSION
    entity_col = "articles_id"
    entity_table = "article"
    time_col = "timestamp"
    target_col = "quantity"
    timedelta = pd.Timedelta(days=1)  # Updated to 1 day
    metrics = [r2, mae, rmse]

    def make_table(self, db: Database, timestamps: "pd.Series[pd.Timestamp]") -> Table:
        transactions = db.table_dict["transactions"].df
        timestamp_df = pd.DataFrame({"timestamp": timestamps})
        article = db.table_dict["article"].df

        df = duckdb.sql(
            f"""
            SELECT
                timestamp,
                articles_id,
                quantity
            FROM
                timestamp_df,
                article,
                (
                    SELECT
                        COALESCE(SUM(Quantity), 0) AS quantity
                    FROM
                        transactions
                    WHERE
                        transactions.articles_id = article.articles_id AND
                        datetime > timestamp AND
                        datetime <= timestamp + INTERVAL '{self.timedelta}'
                )
            """
        ).df()

        return Table(
            df=df,
            fkey_col_to_pkey_table={"articles_id": "article"},
            pkey_col=None,
            time_col="timestamp",
        )




In [65]:
aras_sale_task = ItemQuantityTask(hyper_dataset, cache_dir="D:/Dani/relbench/relbench/cache/hyper_aras390a28432701465676887911s16695")
aras_sale_task

ItemQuantityTask(dataset=TransactionalDataset())

In [66]:
register_task("burger-aras6666666", "aras_sale_task1", ItemQuantityTask)
get_task_names("burger-aras6666666")

['aras_sale_task1']

In [67]:
import numpy as np

from torch.nn import BCEWithLogitsLoss, L1Loss
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("burger-aras6666666")
task = get_task("burger-aras6666666", "aras_sale_task1")


train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

out_channels = 1
loss_fn = L1Loss()
tune_metric = "mae"
higher_is_better = False

Making task table for train split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Making Database object from scratch...
(You can also use `get_dataset(..., download=True)` for datasets prepared by the RelBench team.)
Duplicates found in the 'articles_id' column. Removing duplicates...
Done in 23.64 seconds.
Caching Database object to C:\Users\KN2C\AppData\Local\relbench\relbench\Cache/burger-aras6666666/db...
Done in 1.93 seconds.
Loading Database object from C:\Users\KN2C\AppData\Local\relbench\relbench\Cache/burger-aras6666666/db...
Done in 0.57 seconds.
Done in 29.93 seconds.
Making task table for val split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Done in 0.03 seconds.
Making task table for test split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Loading Database object from C:\Users\KN2C\AppData\Lo

In [68]:
train_table

Table(df=
        timestamp  articles_id  quantity
0      2022-06-11            2     519.0
1      2022-06-11           10      38.0
2      2022-06-11            1     114.0
3      2022-06-11            3       6.0
4      2022-06-11           50      59.0
...           ...          ...       ...
122011 2023-08-09          118       0.0
122012 2022-09-21          118       0.0
122013 2023-01-03          118       0.0
122014 2023-04-08          118       0.0
122015 2023-04-20          121       0.0

[122016 rows x 3 columns],
  fkey_col_to_pkey_table={'articles_id': 'article'},
  pkey_col=None,
  time_col=timestamp)

In [69]:
val_table

Table(df=
     timestamp  articles_id  quantity
0   2024-12-01          115     262.0
1   2024-12-01           38     150.0
2   2024-12-01           67      58.0
3   2024-12-01           25      60.0
4   2024-12-01          114     543.0
..         ...          ...       ...
119 2024-12-01           99       0.0
120 2024-12-01          105       0.0
121 2024-12-01           27       0.0
122 2024-12-01          100       0.0
123 2024-12-01          103       0.0

[124 rows x 3 columns],
  fkey_col_to_pkey_table={'articles_id': 'article'},
  pkey_col=None,
  time_col=timestamp)

In [70]:
test_table

Table(df=
     timestamp  articles_id
0   2024-12-02           33
1   2024-12-02           15
2   2024-12-02           11
3   2024-12-02          112
4   2024-12-02           39
..         ...          ...
119 2024-12-02          100
120 2024-12-02           99
121 2024-12-02           45
122 2024-12-02          101
123 2024-12-02           76

[124 rows x 2 columns],
  fkey_col_to_pkey_table={'articles_id': 'article'},
  pkey_col=None,
  time_col=timestamp)

In [47]:
import os
import math
import numpy as np
from tqdm import tqdm

import torch
import torch_geometric
import torch_frame

# Some book keeping
from torch_geometric.seed import seed_everything

seed_everything(42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # check that it's cuda if you want it to run in reasonable time!
root_dir = "D:/Dani/relbench/relbench/data_ARAS"

cuda


In [71]:
from relbench.modeling.utils import get_stype_proposal

db = dataset.get_db()
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'article': {'articles_id': <stype.numerical: 'numerical'>,
  'art_name': <stype.text_embedded: 'text_embedded'>,
  'price_item': <stype.numerical: 'numerical'>,
  'FoodTypeName': <stype.text_embedded: 'text_embedded'>,
  'group_id': <stype.numerical: 'numerical'>},
 'branches': {'BranchCode': <stype.numerical: 'numerical'>,
  'BranchName': <stype.text_embedded: 'text_embedded'>},
 'customer': {'customers_id': <stype.numerical: 'numerical'>},
 'transactions': {'SaleTime': <stype.timestamp: 'timestamp'>,
  'customers_id': <stype.numerical: 'numerical'>,
  'articles_id': <stype.numerical: 'numerical'>,
  'Quantity': <stype.numerical: 'numerical'>,
  'takhfif': <stype.numerical: 'numerical'>,
  'NetAmount': <stype.numerical: 'numerical'>,
  'Num': <stype.numerical: 'numerical'>,
  'Sales channel': <stype.categorical: 'categorical'>,
  'BranchCode': <stype.categorical: 'categorical'>,
  'price_purchase': <stype.numerical: 'numerical'>,
  'Total_Quantity': <stype.numerical: 'numerical'>,
  

In [74]:
from typing import List, Optional
from sentence_transformers import SentenceTransformer
from torch import Tensor
import torch

class BertPersianTextEmbedding:
    def __init__(self, device: Optional[torch.device] = None):
        # Replace the model with a Persian BERT model
        self.model = SentenceTransformer("HooshvareLab/bert-fa-zwnj-base",  # Example Persian BERT model
            device=device,
        )

    def __call__(self, sentences: List[str]) -> Tensor:
        # Encode the sentences using the Persian BERT model and return as a tensor
        return torch.from_numpy(self.model.encode(sentences))


In [76]:
from torch_frame.config.text_embedder import TextEmbedderConfig
from relbench.modeling.graph import make_pkey_fkey_graph

text_embedder_cfg = TextEmbedderConfig(
    text_embedder=BertPersianTextEmbedding(device=device), batch_size=64
)

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype_dict,  # speficied column types
    text_embedder_cfg=text_embedder_cfg,  # our chosen text encoder
    cache_dir=os.path.join(
        root_dir, f"rel-aras_recom_materialized_cache78s1"
    ),  # store materialized graph for convenience
)

Some weights of BertModel were not initialized from the model checkpoint at HooshvareLab/bert-fa-zwnj-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Embedding raw data in mini-batch: 100%|██████████| 1/1 [00:00<00:00, 10.90it/s]
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\torch_frame\data\stats.py:177: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is

## Sales Forcasting

In [77]:
from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph
from torch_geometric.loader import NeighborLoader

loader_dict = {}

for split, table in [
    ("train", train_table),
    ("val", val_table),
    ("test", test_table),
]:
    table_input = get_node_train_table_input(
        table=table,
        task=task,
    )
    entity_table = table_input.nodes[0]
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[
            128 for i in range(2)
        ],  # we sample subgraphs of depth 2, 128 neighbors per node.
        time_attr="time",
        input_nodes=table_input.nodes,
        input_time=table_input.time,
        transform=table_input.transform,
        batch_size=512,
        temporal_strategy="uniform",
        shuffle=split == "train",
        num_workers=0,
        persistent_workers=False,
    )

In [78]:
from torch.nn import BCEWithLogitsLoss
import copy
from typing import Any, Dict, List

import torch
from torch import Tensor
from torch.nn import Embedding, ModuleDict
from torch_frame.data.stats import StatType
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from torch_geometric.typing import NodeType

from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder


class Model(torch.nn.Module):

    def __init__(
        self,
        data: HeteroData,
        col_stats_dict: Dict[str, Dict[str, Dict[StatType, Any]]],
        num_layers: int,
        channels: int,
        out_channels: int,
        aggr: str,
        norm: str,
        # List of node types to add shallow embeddings to input
        shallow_list: List[NodeType] = [],
        # ID awareness
        id_awareness: bool = False,
    ):
        super().__init__()

        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={
                node_type: data[node_type].tf.col_names_dict
                for node_type in data.node_types
            },
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[
                node_type for node_type in data.node_types if "time" in data[node_type]
            ],
            channels=channels,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )
        self.embedding_dict = ModuleDict(
            {
                node: Embedding(data.num_nodes_dict[node], channels)
                for node in shallow_list
            }
        )

        self.id_awareness_emb = None
        if id_awareness:
            self.id_awareness_emb = torch.nn.Embedding(1, channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.head.reset_parameters()
        for embedding in self.embedding_dict.values():
            torch.nn.init.normal_(embedding.weight, std=0.1)
        if self.id_awareness_emb is not None:
            self.id_awareness_emb.reset_parameters()

    def forward(
        self,
        batch: HeteroData,
        entity_table: NodeType,
    ) -> Tensor:
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )

        return self.head(x_dict[entity_table][: seed_time.size(0)])

    def forward_dst_readout(
        self,
        batch: HeteroData,
        entity_table: NodeType,
        dst_table: NodeType,
    ) -> Tensor:
        if self.id_awareness_emb is None:
            raise RuntimeError(
                "id_awareness must be set True to use forward_dst_readout"
            )
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)
        # Add ID-awareness to the root node
        x_dict[entity_table][: seed_time.size(0)] += self.id_awareness_emb.weight

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
        )

        return self.head(x_dict[dst_table])


model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=2,
    channels=128,
    out_channels=1,
    aggr="sum",
    norm="batch_norm",
).to(device)


# if you try out different RelBench tasks you will need to change these
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
epochs = 10

In [79]:
def train() -> float:
    model.train()

    loss_accum = count_accum = 0
    for batch in tqdm(loader_dict["train"]):
        batch = batch.to(device)

        optimizer.zero_grad()
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred

        loss = loss_fn(pred.float(), batch[entity_table].y.float())
        loss.backward()
        optimizer.step()

        loss_accum += loss.detach().item() * pred.size(0)
        count_accum += pred.size(0)

    return loss_accum / count_accum


@torch.no_grad()
def test(loader: NeighborLoader) -> np.ndarray:
    model.eval()

    pred_list = []
    for batch in loader:
        batch = batch.to(device)
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred
        pred_list.append(pred.detach().cpu())
    return torch.cat(pred_list, dim=0).numpy()

In [80]:
state_dict = None
best_val_metric = -math.inf if higher_is_better else math.inf
for epoch in range(1, epochs + 1):
    train_loss = train()
    val_pred = test(loader_dict["val"])
    val_metrics = task.evaluate(val_pred, val_table)
    print(f"Epoch: {epoch:02d}, Train loss: {train_loss}, Val metrics: {val_metrics}")

    if (higher_is_better and val_metrics[tune_metric] > best_val_metric) or (
        not higher_is_better and val_metrics[tune_metric] < best_val_metric
    ):
        best_val_metric = val_metrics[tune_metric]
        state_dict = copy.deepcopy(model.state_dict())


model.load_state_dict(state_dict)
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, val_table)
print(f"Best Val metrics: {val_metrics}")

test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 239/239 [12:37<00:00,  3.17s/it]
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Epoch: 01, Train loss: 37.69292229753199, Val metrics: {'r2': 0.03493294541562375, 'mae': np.float64(146.36614358365054), 'rmse': np.float64(396.3593117610412)}


100%|██████████| 239/239 [12:34<00:00,  3.16s/it]
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Epoch: 02, Train loss: 21.65171950425559, Val metrics: {'r2': 0.19862400221263743, 'mae': np.float64(134.07854018192137), 'rmse': np.float64(361.1839855336255)}


100%|██████████| 239/239 [12:32<00:00,  3.15s/it]
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


...

100%|██████████| 239/239 [12:27<00:00,  3.13s/it]
c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Epoch: 10, Train loss: 9.566788848478776, Val metrics: {'r2': 0.2832486120517418, 'mae': np.float64(131.95684561614067), 'rmse': np.float64(341.5816971216677)}


c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Best Val metrics: {'r2': 0.38487770623360795, 'mae': np.float64(124.16544886390048), 'rmse': np.float64(316.4397657434665)}
Best test metrics: {'r2': 0.3910561903229308, 'mae': np.float64(123.2035786242735), 'rmse': np.float64(313.1765912692561)}


c:\Users\KN2C\miniconda3\envs\kamal\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [81]:
test_pred

array([2.51565765e+02, 6.56884308e+01, 1.24683449e+02, 6.14008865e+01,
       5.15306702e+01, 7.19315643e+01, 1.60778160e+01, 2.58031120e+01,
       3.93180418e+00, 1.21712065e+00, 3.01784943e+02, 8.19246521e+01,
       1.67903900e+02, 4.34114685e+01, 6.80621109e+01, 1.61858606e+00,
       4.77440977e+00, 1.21253557e+01, 1.70946465e+01, 4.05860633e-01,
       5.77699661e+01, 8.24392761e+02, 2.78200378e+01, 6.70544434e+01,
       2.77024994e+02, 3.52547407e+00, 1.33501768e+00, 9.25759983e+00,
       1.85084496e+01, 9.26339722e+00, 2.76230488e+01, 4.39467621e+01,
       6.59616590e-02, 2.22136421e+01, 1.31720734e+02, 6.99099350e+01,
       7.98131943e+01, 4.22165871e+00, 7.88564148e+01, 4.80810394e+01,
       3.31198692e+01, 2.73211432e+00, 1.46072865e+01, 1.56750965e+01,
       1.78219280e+01, 1.43912482e+00, 1.17137575e+00, 2.29859066e+00,
       6.62366152e-02, 4.08372974e+00, 2.51310229e-01, 1.63516769e+02,
       1.02188576e+02, 9.03887863e+01, 8.33364014e+02, 3.04736500e+01,
      

In [86]:
customer_ids = test_table.df["articles_id"].values  # Extract the customer_id column
predicted_labels = test_pred  # This contains the model's predicted labels

# Now zip customer_ids with the predictions
results = list(zip(customer_ids, predicted_labels))

# If you'd like to show or process the results, you can format them as a dataframe
results_df = pd.DataFrame(results, columns=["articles_id", "predicted_sale"])

# You can now inspect the predictions
print(results_df)

     articles_id  predicted_sale
0             33      251.565765
1             15       65.688431
2             11      124.683449
3            112       61.400887
4             39       51.530670
..           ...             ...
119          100        0.063264
120           99        0.053299
121           45        0.063780
122          101        0.052302
123           76        0.063822

[124 rows x 2 columns]


In [90]:
# Assuming `task.target_col` contains the name of the label column (e.g., "quantity")
test_labels = test_table.df[task.target_col].values


KeyError: 'quantity'

In [88]:
true_values = test_table.df

In [93]:
test_table = task.get_table(split="test", mask_input_cols=False)

# Convert the table to a pandas DataFrame
test_df = test_table.df
test_df["predicted_sale"] = results_df["predicted_sale"]
# Save the DataFrame to a CSV file
csv_file_path = "test_table_output.csv"
test_df.to_csv(csv_file_path, index=False)

In [92]:
test_table

Table(df=
     timestamp  articles_id  quantity
0   2024-12-02           33     957.0
1   2024-12-02           15     204.0
2   2024-12-02           11     465.0
3   2024-12-02          112     209.0
4   2024-12-02           39     169.0
..         ...          ...       ...
119 2024-12-02          100       0.0
120 2024-12-02           99       0.0
121 2024-12-02           45       0.0
122 2024-12-02          101       0.0
123 2024-12-02           76       0.0

[124 rows x 3 columns],
  fkey_col_to_pkey_table={'articles_id': 'article'},
  pkey_col=None,
  time_col=timestamp)